# 99 CheatSheet and 40 Practice Questions

**Purpose:** Quick reference cheat sheet for Databricks ML Professional topics and 40 practice questions with concise answers to help you prepare and validate knowledge.

Run this notebook in Databricks Repos as a study aid. Use the questions for timed practice and the cheat sheet for quick reminders during labs.

## Cheat Sheet — Key Concepts (condensed)

- **Point‑in‑time join:** Always enforce `feature_ts <= label_ts`. Use window functions and `row_number()` to pick the latest feature row per label event.
- **Delta Lake:** ACID transactions, time travel, `MERGE INTO` for upserts, partition by high‑cardinality keys carefully, use `OPTIMIZE` and `VACUUM` for maintenance.
- **Feature materialization:** Materialize offline features to Delta; for online lookups use a feature store or low‑latency store; maintain feature timestamps and keys.
- **MLflow basics:** `mlflow.start_run()`, `mlflow.log_param()`, `mlflow.log_metric()`, `mlflow.sklearn.log_model()`, `MlflowClient()` for registry operations.
- **Model Registry:** Register model, create versions, transition stages (`Staging`, `Production`), add tags and descriptions, automate promotions via API.
- **Serving canary:** Route small % of traffic to new model, compare metrics, rollback if thresholds breached; capture latency, error rate, prediction drift.
- **Drift detection:** Use KS test for numeric features, JS divergence for categorical distributions, monitor metrics over windows and require sustained signals.
- **Databricks Jobs:** Use job clusters or existing clusters, schedule notebooks or Python tasks, trigger via REST API, secure tokens in secrets.
- **Secrets:** Use Databricks secret scopes; never hardcode tokens; access via `dbutils.secrets.get(scope, key)`.
- **CI/CD:** Use GitHub Actions to run smoke tests, call Databricks Jobs API, store tokens in GitHub Secrets, require PR reviews for `main`.


## 40 Practice Questions and Answers

Use these for quick recall. Try answering before reading the provided concise answer.

1. **Q:** What is the primary risk when joining features to labels without point‑in‑time logic?
**A:** Label leakage — using future information (features with timestamps after the label) inflates performance and invalidates models.

2. **Q:** How do you implement a point‑in‑time join in Spark/Delta?
**A:** Join labels to events with condition `event.feature_ts <= label.label_ts`, use a window partitioned by label keys and `row_number()` ordered by `feature_ts` desc, then select `rn=1`.

3. **Q:** What are the core benefits of Delta Lake?
**A:** ACID transactions, scalable metadata, time travel, schema enforcement/evolution, and efficient upserts via `MERGE`.

4. **Q:** When should you use `MERGE INTO` vs `INSERT`?
**A:** Use `MERGE` for upserts (insert/update/delete) when synchronizing incremental data; `INSERT` for append‑only writes.

5. **Q:** What is the recommended way to store secrets for Databricks jobs and notebooks?
**A:** Use Databricks secret scopes (Databricks‑backed or external key vault) and access via `dbutils.secrets.get()`.

6. **Q:** How do you register a model in MLflow programmatically?
**A:** Use `MlflowClient.create_registered_model(name)` then `create_model_version(name, source, run_id)` to create a version from a run artifact.

7. **Q:** What metadata should you attach to a registered model?
**A:** Training dataset snapshot, feature lineage, hyperparameters, metrics, model signature, input example, and tags for traceability.

8. **Q:** How do you add a model signature in MLflow?
**A:** Use `infer_signature(input_df, model.predict(input_df))` or build `ModelSignature` and pass `signature=` to `mlflow.sklearn.log_model()`.

9. **Q:** What is a safe canary rollout percentage to start with?
**A:** Typically small (1–5%) to limit exposure; choose based on traffic volume and risk tolerance.

10. **Q:** Name two statistical tests for numeric distribution drift.
**A:** Kolmogorov–Smirnov (KS) test and population mean/variance tests; use KS for distribution shape differences.

11. **Q:** How do you measure categorical distribution drift?
**A:** Use Jensen–Shannon divergence, Chi‑square test, or compare normalized frequency vectors.

12. **Q:** What is the Databricks recommended way to trigger a job from an external CI system?
**A:** Call the Databricks Jobs REST API (`/api/2.1/jobs/run-now`) using a workspace PAT stored in secrets.

13. **Q:** How do you avoid committing secrets to Git?
**A:** Never store secrets in code; use secret scopes, environment variables in CI, and Git ignore for local config files.

14. **Q:** What is the purpose of `dbutils.secrets`?
**A:** Securely retrieve secrets stored in Databricks secret scopes inside notebooks and jobs without exposing values in logs.

15. **Q:** When should you enable Git LFS for a repo?
**A:** When storing large binary files (models, large datasets, notebooks with outputs) to avoid bloating the Git history.

16. **Q:** How do you perform an incremental feature materialization?
**A:** Track last processed watermark, compute new aggregates for the window since watermark, upsert into feature table using `MERGE`.

17. **Q:** What is the difference between offline and online features?
**A:** Offline features are precomputed for training (Delta tables); online features are low‑latency lookups for serving (feature store or key‑value store).

18. **Q:** How do you validate that a training set has no leakage programmatically?
**A:** Assert `feature_ts <= label_ts` for all rows, compare counts to expected labels, and run unit tests on synthetic known cases.

19. **Q:** What are common MLflow experiment best practices?
**A:** Use descriptive experiment names, log params/metrics/artifacts consistently, attach tags, and store input examples and signatures.

20. **Q:** How do you promote a model version to Production via API?
**A:** Use `MlflowClient.transition_model_version_stage(name, version, stage='Production')` or call the Model Registry REST API.

21. **Q:** What should a smoke test for a promoted model include?
**A:** Load model from registry, run a few representative predictions, verify output shape and basic accuracy or sanity checks.

22. **Q:** How do you capture model lineage for reproducibility?
**A:** Log training data snapshot (hash or path), feature engineering code, environment (conda/pip), hyperparameters, and run id in MLflow.

23. **Q:** What is the role of `mlflow.sklearn.log_model()` vs `mlflow.pyfunc.log_model()`?
**A:** `mlflow.sklearn.log_model()` logs scikit‑learn models with flavor metadata; `pyfunc` provides a generic Python function interface for serving.

24. **Q:** How do you handle late arriving events in feature pipelines?
**A:** Use backfills, reprocessing windows, watermarking, or store raw events and recompute affected features for impacted labels.

25. **Q:** What metrics are important to monitor for a serving endpoint?
**A:** Latency, error rate, throughput, prediction distribution, model confidence, and business metrics (e.g., conversion rate).

26. **Q:** How do you implement automated rollback for a failed canary?
**A:** Detect failure via alerts, call serving API to revert traffic routing or redeploy previous model version, and notify stakeholders.

27. **Q:** What is a model signature and why is it useful?
**A:** A signature defines input/output schema for a model; it helps validate inputs at serving time and prevents schema mismatches.

28. **Q:** How do you test feature correctness locally before running on full data?
**A:** Use small synthetic datasets with known expected outputs and unit tests that assert feature values and edge cases.

29. **Q:** What is the recommended partitioning strategy for large Delta feature tables?
**A:** Partition by low‑cardinality, high‑selectivity columns (e.g., date) and avoid high‑cardinality keys like user_id unless necessary; use Z‑ordering for selective queries.

30. **Q:** How do you secure access to Databricks REST APIs in CI workflows?
**A:** Store workspace PAT in CI secrets, restrict token scopes, and rotate tokens regularly; use least privilege.

31. **Q:** What is the difference between Databricks Repos and Workspace files?
**A:** Repos provide Git integration for notebooks and files with two‑way sync; Workspace stores notebooks but lacks native Git versioning.

32. **Q:** How do you measure concept drift vs data drift?
**A:** Data drift measures input distribution changes; concept drift measures changes in the relationship between inputs and labels (model performance degradation).

33. **Q:** What is a practical thresholding approach for drift alerts?
**A:** Use statistical test p‑values and effect sizes, require sustained breaches over multiple windows, and combine multiple signals to reduce false positives.

34. **Q:** How do you register and use a feature table in Databricks Feature Store?
**A:** Use Feature Store APIs to create a feature table, materialize features to Delta, and use `FeatureStoreClient` to read features for training and serving.

35. **Q:** What are common causes of flaky CI tests for notebooks and how to mitigate?
**A:** Environment differences, network calls, non‑deterministic randomness; mitigate by pinning seeds, mocking external services, and using stable test datasets.

36. **Q:** How do you perform a reproducible model training environment capture?
**A:** Log conda/pip environment (`mlflow.log_artifact` of `conda.yaml`), use Docker images, or record runtime and library versions in MLflow tags.

37. **Q:** What is the recommended way to store large model artifacts?
**A:** Use object storage (S3/ADLS/DBFS) and Git LFS for large files; MLflow stores artifacts in configured artifact stores.

38. **Q:** How do you test a Databricks Job trigger from GitHub Actions?
**A:** Create a workflow step that calls Databricks Jobs REST API using secrets for host and token, and assert the API returns a run id.

39. **Q:** What are the advantages of using MLflow Model Registry over ad‑hoc artifact storage?
**A:** Centralized model lifecycle management, versioning, stage transitions, annotations, and integration with serving and governance workflows.

40. **Q:** How should you document a lab or notebook to maximize reproducibility for others?
**A:** Include purpose, prerequisites, runtime and library versions, secrets required, step‑by‑step run order, expected outputs, and troubleshooting tips.


## How to use these questions
- **Self‑test:** Hide the answers and time yourself answering each question.  
- **Flashcards:** Convert each Q/A into flashcards for spaced repetition.  
- **Hands‑on:** For each question, open the corresponding notebook or lab and run the steps to reinforce practical skills.  
- **Extend:** Add more questions specific to your environment (cloud provider, runtime versions, org policies).

## Next steps
- Import this notebook into Databricks Repos and use it as a study checkpoint.  
- If you want, I can convert these questions into a printable checklist, a GitHub issue template for study sessions, or generate a set of unit tests that validate key lab assertions.  
Pick one and I will generate it next.